In [ ]:
GPU="0"
num_GPUs = 1
gen_image_batch_size=256

In [ ]:
import os
os.environ['PATH'] += ':/home/temp0/anaconda3/envs/edm2/bin'

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def show_images_from_dir(folder_path, num_images=4, start_seed=42):
    """
    顯示指定資料夾中的圖片（根據檔名排序，從指定種子碼開始），以 2x2 格顯示。

    Args:
        folder_path (str): 圖片所在資料夾路徑
        num_images (int): 要顯示的圖片數量（預設 4）
        start_seed (int): 從第幾個種子（圖片）開始（根據檔名排序）
    """
    if not os.path.isdir(folder_path):
        print(f"[錯誤] 資料夾不存在：{folder_path}")
        return

    # 過濾圖片檔案，並根據檔名中的數字排序
    image_files = sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ], key=lambda x: int(''.join(filter(str.isdigit, os.path.basename(x))) or 0))

    # 找到起始 index
    start_index = next((i for i, f in enumerate(image_files)
                        if int(''.join(filter(str.isdigit, os.path.basename(f))) or 0) >= start_seed), None)

    if start_index is None:
        print(f"[警告] 找不到 seed >= {start_seed} 的圖片。")
        return

    subset = image_files[start_index:start_index + num_images]
    if not subset:
        print(f"[警告] 從 seed {start_seed} 起沒有足夠圖片可顯示。")
        return

    # 顯示 2x2 圖片
    rows = cols = 2
    plt.figure(figsize=(8, 8))
    for i, img_path in enumerate(subset):
        img = Image.open(img_path)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
w=2.1
k=3
w_interval_low_steps = 10
w_interval_low_peak_steps = 15
w_interval_high_peak_steps =20
w_interval_high_steps = 25
num_images = 30000
dirname=f'/data/guidance-team/out/2025_0728_Random_DDG_w={w}_k={k}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
      --preset=edm2-img512-s-guid-fid \
      --outdir=/{dirname} \
      --subdirs \
      --seeds=0-{num_images+10} \
    --sampler=random_ddg \
    --guidance_scheduler=trapezoid_scheduler \
    --debug=False \
    --guidance={w} \
--k_average={k} \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--heun_guid=False \
--full_random=True \
--heun_guid=False \
--batch=128
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./base/calculate_metrics.py -- calc \
      --images=/{dirname} \
      --ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
      --num={num_images} \
      --metrics='fid' \
      --batch=4
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}

In [ ]:
w=2.1
k=3
w_interval_low_steps = 10
w_interval_low_peak_steps = 15
w_interval_high_peak_steps =20
w_interval_high_steps = 25
num_images = 50000
dirname=f'/data/guidance-team/out/2025_0728_Random_DDG_w={w}_k={k}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
      --preset=edm2-img512-s-guid-fid \
      --outdir=/{dirname} \
      --subdirs \
      --seeds=0-{num_images+10} \
    --sampler=random_ddg \
    --guidance_scheduler=trapezoid_scheduler \
    --debug=False \
    --guidance={w} \
--k_average={k} \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--heun_guid=False \
--full_random=True \
--heun_guid=False \
--batch=512
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./base/calculate_metrics.py -- calc \
      --images=/{dirname} \
      --ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
      --num={num_images} \
      --metrics='fid' \
      --batch=4
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}

In [ ]:
n=5
k=1
w=2.1
w_interval_low_steps = 14
w_interval_low_peak_steps = 15
w_interval_high_peak_steps = 19
w_interval_high_steps = 30
num_images = 300
dirname=f'/data/guidance-team/out/2025_0728_MAP_DDG_w={w}_k={k}_n={n},trapezoid,{w_interval_low_steps},{w_interval_low_peak_steps},{w_interval_high_peak_steps},{w_interval_high_steps}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_neg_ddg \
--guidance_scheduler=trapezoid_scheduler \
--debug=False \
--guidance={w} \
--n_random={n} \
--k_average={k} \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_low_peak_steps={w_interval_low_peak_steps} \
--w_interval_high_peak_steps={w_interval_high_peak_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--heun_guid=False \
--batch={gen_image_batch_size} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_pure_first/model085400.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 2 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./base/calculate_metrics.py calc \
      --images=/{dirname} \
      --ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
      --num={num_images} \
      --metrics='fid' \
      --batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000